In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/data.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 64 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   4800 non-null   object 
 1   x0      4800 non-null   float64
 2   y0      4800 non-null   float64
 3   z0      4800 non-null   float64
 4   x1      4800 non-null   float64
 5   y1      4800 non-null   float64
 6   z1      4800 non-null   float64
 7   x2      4800 non-null   float64
 8   y2      4800 non-null   float64
 9   z2      4800 non-null   float64
 10  x3      4800 non-null   float64
 11  y3      4800 non-null   float64
 12  z3      4800 non-null   float64
 13  x4      4800 non-null   float64
 14  y4      4800 non-null   float64
 15  z4      4800 non-null   float64
 16  x5      4800 non-null   float64
 17  y5      4800 non-null   float64
 18  z5      4800 non-null   float64
 19  x6      4800 non-null   float64
 20  y6      4800 non-null   float64
 21  z6      4800 non-null   float64
 22  

In [3]:
df.head()

,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20
0,A,0.850228,0.550703,-5.601127e-07,0.780329,0.526167,-0.017441,0.719578,0.456316,-0.023337,...,-0.006061,0.833324,0.336230,-0.028391,0.831558,0.390031,-0.020616,0.836817,0.429634,-0.006872
1,A,0.850415,0.553021,-5.687378e-07,0.778991,0.523142,-0.015536,0.720142,0.455045,-0.021170,...,-0.005319,0.833279,0.336285,-0.026664,0.830972,0.388208,-0.019361,0.836294,0.428583,-0.006194
2,A,0.847148,0.550497,-5.636708e-07,0.778737,0.523869,-0.016781,0.720492,0.454723,-0.022722,...,-0.007982,0.834024,0.339281,-0.030549,0.831053,0.391668,-0.023533,0.835561,0.432826,-0.010354
3,A,0.849373,0.553090,-5.629355e-07,0.778276,0.522691,-0.016358,0.719755,0.455217,-0.021801,...,-0.003113,0.834957,0.336697,-0.025234,0.831425,0.389026,-0.018396,0.835111,0.429615,-0.005336
4,A,0.849822,0.552474,-5.676418e-07,0.779409,0.524942,-0.016822,0.720351,0.456630,-0.022638,...,-0.006129,0.833818,0.338993,-0.028827,0.831514,0.393562,-0.021462,0.836137,0.432677,-0.008101


- Chia dữ liệu gốc thành 2 loại: Biến độc lập (X) và biến phụ thuộc (y)

In [4]:
X = df.drop('label', axis=1)
y = df['label']

print(X.shape)

(4800, 63)


- Xây dựng hàm để đưa dữ liệu về hệ quy chiếu chung --> giúp mô hình nhận diện chính xác tay dù ở vị trí / khoảng cách nào.

In [5]:
def normalize_landmarks(row):
    coords = row.values.reshape(21,3)
    
    coords = coords - coords[0]
    
    max_val = np.max(np.abs(coords))
    if max_val != 0:
        coords = coords / max_val
        
    return coords.flatten()

In [6]:
X_norm = np.array([normalize_landmarks(row) for _, row in X.iterrows()])

print(X_norm.shape)

(4800, 63)


- Xây dựng hàm tính toán khoảng cách Euclid giữa các khớp ngón tay --> mô hình hiểu nhanh hơn trạng thái của bàn tay

In [7]:
def compute_distances(coords):
    coords = coords.reshape(21, 3)

    def dist(i, j):
        return np.linalg.norm(coords[i] - coords[j])

    features = []

    pairs = [
        (4,8), (8,12), (12,16), (16,20),  # giữa các ngón
        (0,8), (0,12), (0,16), (0,20)     # cổ tay đến đầu ngón
    ]

    for i, j in pairs:
        features.append(dist(i, j))

    return np.array(features)

- Tổng hợp dữ liệu đặc trưng

In [8]:
X_features = []

for row in X_norm:
    distances = compute_distances(row)
    combined = np.concatenate([row, distances])
    X_features.append(combined)
    
X_features = np.array(X_features)

print(X_features.shape)

(4800, 71)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_features,
                                                    y,
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=42)

print("Số mẫu huấn luyện: ", len(X_train))
print("Số mẫu kiểm thử: ", len(X_test))

Số mẫu huấn luyện:  3840
Số mẫu kiểm thử:  960


- Chuẩn hóa phân phối và lưu

In [10]:
from sklearn.preprocessing import StandardScaler
import joblib
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(X_train, '../data/X_train.pkl')
joblib.dump(X_test, '../data/X_test.pkl')
joblib.dump(y_train, '../data/y_train.pkl')
joblib.dump(y_test, '../data/y_test.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']